In [2]:
import pandas as pd
import numpy as np
import os
import joblib

from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score
)

# ==========================================
# 1. LOAD PROCESSED DATA
# ==========================================

X_train = pd.read_csv("../data/processed/X_train.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")

y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

print("===== DATA LOADED =====")
print("Training features:", X_train.shape)
print("Testing features :", X_test.shape)
print("Training labels  :", y_train.shape)
print("Testing labels   :", y_test.shape)


# ==========================================
# 2. PREPARE TARGET FOR XGBOOST
# ==========================================

# Dataset labels are:
# 1 = Normal
# 2 = Suspect
# 3 = Pathological
#
# XGBoost uses class labels starting at 0,
# so convert 1,2,3 -> 0,1,2

y_train_xgb = y_train.astype(int) - 1
y_test_xgb = y_test.astype(int) - 1


# ==========================================
# 3. CREATE XGBOOST MODEL
# ==========================================

print("\n===== CREATING XGBOOST =====")

model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softmax",
    num_class=3,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)


# ==========================================
# 4. TRAIN
# ==========================================

print("\n===== TRAINING XGBOOST =====")

model.fit(
    X_train,
    y_train_xgb
)

print("Training complete!")


# ==========================================
# 5. PREDICTIONS
# ==========================================

y_pred_xgb = model.predict(X_test)

# Convert 0,1,2 back to 1,2,3
y_pred = y_pred_xgb + 1


# ==========================================
# 6. EVALUATION
# ==========================================

macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro"
)

print("\n===== XGBOOST RESULTS =====")

print("\nMacro F1 Score:")
print(round(macro_f1, 4))

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            "Normal",
            "Suspect",
            "Pathological"
        ]
    )
)

print("\nConfusion Matrix:")

cm = confusion_matrix(
    y_test,
    y_pred
)

print(cm)


# ==========================================
# 7. FEATURE IMPORTANCE
# ==========================================

importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": model.feature_importances_
})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

print("\n===== TOP 15 FEATURES =====")

print(
    importance.head(15).to_string(index=False)
)


# ==========================================
# 8. SAVE MODEL
# ==========================================

os.makedirs("../models", exist_ok=True)

joblib.dump(
    model,
    "../models/xgboost.pkl"
)


# ==========================================
# 9. SAVE FEATURE IMPORTANCE
# ==========================================

os.makedirs("../results", exist_ok=True)

importance.to_csv(
    "../results/xgboost_feature_importance.csv",
    index=False
)


# ==========================================
# 10. FINAL MESSAGE
# ==========================================

print("\n===== SAVED =====")
print("Model:")
print("../models/xgboost.pkl")

print("\nFeature importance:")
print("../results/xgboost_feature_importance.csv")

===== DATA LOADED =====
Training features: (1692, 41)
Testing features : (424, 41)
Training labels  : (1692,)
Testing labels   : (424,)

===== CREATING XGBOOST =====

===== TRAINING XGBOOST =====
Training complete!

===== XGBOOST RESULTS =====

Macro F1 Score:
0.9898

Classification Report:
              precision    recall  f1-score   support

      Normal       0.99      1.00      1.00       330
     Suspect       1.00      0.95      0.97        59
Pathological       1.00      1.00      1.00        35

    accuracy                           0.99       424
   macro avg       1.00      0.98      0.99       424
weighted avg       0.99      0.99      0.99       424


Confusion Matrix:
[[330   0   0]
 [  3  56   0]
 [  0   0  35]]

===== TOP 15 FEATURES =====
Feature  Importance
     LD    0.180031
   SUSP    0.170380
     FS    0.143073
  CLASS    0.113858
      E    0.093640
      A    0.046232
      B    0.031884
   MSTV    0.020734
   ASTV    0.020209
     DE    0.018360
     AC    0.